# Greedy decoding, beam search, and reinforcement learning perplexities

The previous toy version used letter strings, which made the outputs hard to interpret. This version uses a tiny sentence generator so the decoding examples are readable.

We compare three sequence-generation strategies with **perplexity** under the same probabilistic evaluator:

$$\text{perplexity}=\exp\left(-\frac{1}{N}\sum_{i=1}^N \log p(x_i \mid x_{<i})\right).$$

Lower perplexity means the evaluator assigns higher probability to the generated tokens. The goal is not to claim that perplexity equals human quality; it is to show how greedy decoding, beam search, and reinforcement learning can be compared with a repeatable likelihood-based score.


In [1]:
import math
import random

random.seed(7)

EOS = "<eos>"
START = ()

# A tiny prefix-conditioned language model. Each key is the current prefix, and
# each value is a next-token distribution. All complete paths form sensible toy
# sentences, so generated outputs can be inspected as language rather than as
# arbitrary character strings.
MODEL = {
    START: {"the": 0.55, "a": 0.45},
    ("the",): {"robot": 0.80, "child": 0.20},
    ("the", "robot"): {"computed": 0.34, "painted": 0.33, "danced": 0.33},
    ("the", "robot", "computed"): {EOS: 0.90, "quietly": 0.10},
    ("the", "robot", "computed", "quietly"): {EOS: 1.00},
    ("the", "robot", "painted"): {"a": 0.72, EOS: 0.28},
    ("the", "robot", "painted", "a"): {"mural": 0.92, "portrait": 0.08},
    ("the", "robot", "painted", "a", "mural"): {EOS: 0.96},
    ("the", "robot", "painted", "a", "portrait"): {EOS: 0.90},
    ("the", "robot", "danced"): {"gracefully": 0.80, EOS: 0.20},
    ("the", "robot", "danced", "gracefully"): {EOS: 0.95},
    ("the", "child"): {"read": 0.72, "laughed": 0.28},
    ("the", "child", "read"): {"a": 0.95, EOS: 0.05},
    ("the", "child", "read", "a"): {"book": 0.95},
    ("the", "child", "read", "a", "book"): {EOS: 0.97},
    ("the", "child", "laughed"): {"softly": 0.90, EOS: 0.10},
    ("the", "child", "laughed", "softly"): {EOS: 0.96},
    ("a",): {"dog": 0.96, "bird": 0.04},
    ("a", "dog"): {"chased": 0.96, "watched": 0.04},
    ("a", "dog", "chased"): {"the": 0.96, "a": 0.04},
    ("a", "dog", "chased", "the"): {"ball": 0.96, "cat": 0.04},
    ("a", "dog", "chased", "the", "ball"): {EOS: 0.96},
    ("a", "dog", "chased", "the", "cat"): {EOS: 0.92},
    ("a", "dog", "chased", "a"): {"stick": 0.90},
    ("a", "dog", "chased", "a", "stick"): {EOS: 0.95},
    ("a", "dog", "watched"): {"the": 0.92},
    ("a", "dog", "watched", "the"): {"moon": 0.92},
    ("a", "dog", "watched", "the", "moon"): {EOS: 0.95},
    ("a", "bird"): {"sang": 0.90},
    ("a", "bird", "sang"): {"softly": 0.90},
    ("a", "bird", "sang", "softly"): {EOS: 0.95},
}

def next_distribution(prefix):
    return MODEL[tuple(prefix)]

def draw_from(dist):
    r = random.random()
    total = 0.0
    for token, prob in dist.items():
        total += prob
        if r <= total:
            return token
    return token

def render(sequence):
    return " ".join(token for token in sequence if token != EOS)


## Perplexity helper

Perplexity includes the end-of-sequence token. This matters because a decoder that stops early is making a probability-bearing decision, just like a decoder that chooses another word.


In [2]:
def sequence_log_probability(sequence):
    prefix = []
    logp = 0.0
    for token in sequence:
        p = next_distribution(prefix)[token]
        logp += math.log(p)
        if token == EOS:
            break
        prefix.append(token)
    return logp

def corpus_metrics(corpus):
    token_count = sum(len(seq) for seq in corpus)
    total_logp = sum(sequence_log_probability(seq) for seq in corpus)
    nll = -total_logp / token_count
    return {"tokens": token_count, "nll": nll, "perplexity": math.exp(nll)}

def evaluate_generation(label, generated):
    m = corpus_metrics(generated)
    print(f"{label:16s} perplexity={m['perplexity']:.3f}  nll/token={m['nll']:.3f}")
    for seq in generated[:4]:
        print("  ", render(seq))
    print()
    return m


## Decode with greedy search and beam search

Greedy decoding chooses the most likely next token at the current prefix. Beam search keeps several partial candidates, so it can prefer a slightly less likely first word when that choice leads to a much stronger complete sentence.


In [3]:
def greedy_decode():
    prefix = []
    out = []
    while True:
        dist = next_distribution(prefix)
        token = max(dist, key=dist.get)
        out.append(token)
        if token == EOS:
            return tuple(out)
        prefix.append(token)

def beam_search_decode(beam_width=3):
    beam = [((), 0.0)]
    complete = []
    while beam:
        candidates = []
        for seq, score in beam:
            if seq and seq[-1] == EOS:
                complete.append((seq, score))
                continue
            prefix = [token for token in seq if token != EOS]
            for token, prob in next_distribution(prefix).items():
                candidates.append((seq + (token,), score + math.log(prob)))
        candidates.sort(key=lambda item: item[1], reverse=True)
        beam = candidates[:beam_width]
    complete.sort(key=lambda item: item[1], reverse=True)
    return complete[0][0]

greedy_sequence = greedy_decode()
beam_sequence = beam_search_decode(beam_width=3)

print("Greedy:", render(greedy_sequence), f"logp={sequence_log_probability(greedy_sequence):.3f}")
print("Beam:  ", render(beam_sequence), f"logp={sequence_log_probability(beam_sequence):.3f}")


Greedy: the robot computed logp=-2.005
Beam:   a dog chased the ball logp=-1.003


## A tiny reinforcement-learning policy

For an educational notebook, we do not need a neural network to explain the evaluation idea. We enumerate the complete candidate sentences and assign each sentence a reward equal to its average log probability under the evaluator. An entropy-regularized policy then samples complete sentences with probability proportional to `exp(reward / temperature)`.

This is the tabular analogue of reinforcement learning that optimizes a sequence-level reward while retaining exploration. It should produce better perplexity than the greedy sentence, but worse perplexity than the single best beam-search sentence.


In [4]:
def enumerate_sentences(prefix=()):
    sentences = []
    for token in next_distribution(prefix).keys():
        candidate = prefix + (token,)
        if token == EOS:
            sentences.append(candidate)
        else:
            sentences.extend(enumerate_sentences(candidate))
    return sentences

def average_log_probability(sequence):
    return sequence_log_probability(sequence) / len(sequence)

all_sentences = enumerate_sentences()
scored_sentences = sorted(
    [(seq, average_log_probability(seq), math.exp(-average_log_probability(seq))) for seq in all_sentences],
    key=lambda item: item[1],
    reverse=True,
)

print("Top candidate sentences by evaluator perplexity:")
for seq, avg_logp, ppl in scored_sentences[:5]:
    print(f"  perplexity={ppl:.3f}  {render(seq)}")

def rl_sentence_policy(temperature=0.18):
    weights = [math.exp(avg_logp / temperature) for _, avg_logp, _ in scored_sentences]
    total = sum(weights)
    return [(seq, weight / total) for (seq, _, _), weight in zip(scored_sentences, weights)]

def sample_rl_sentence(temperature=0.18):
    return draw_from(dict(rl_sentence_policy(temperature)))


Top candidate sentences by evaluator perplexity:
  perplexity=1.182  a dog chased the ball
  perplexity=1.487  the robot painted a mural
  perplexity=1.554  the robot danced gracefully
  perplexity=1.560  the child read a book
  perplexity=1.651  the robot computed


## Perplexity comparison

The comparison below uses repeated greedy and beam outputs because those decoders are deterministic. The RL policy is stochastic, so its sample includes several high-reward sentences instead of just one sentence.


In [5]:
sample_count = 40
greedy_samples = [greedy_sequence for _ in range(sample_count)]
beam_samples = [beam_sequence for _ in range(sample_count)]
rl_samples = [sample_rl_sentence(temperature=0.18) for _ in range(sample_count)]

comparison = {
    "Greedy": evaluate_generation("Greedy", greedy_samples),
    "Beam search": evaluate_generation("Beam search", beam_samples),
    "RL policy": evaluate_generation("RL policy", rl_samples),
}

assert comparison["Beam search"]["perplexity"] <= comparison["RL policy"]["perplexity"] <= comparison["Greedy"]["perplexity"]


Greedy           perplexity=1.651  nll/token=0.501
   the robot computed
   the robot computed
   the robot computed
   the robot computed

Beam search      perplexity=1.182  nll/token=0.167
   a dog chased the ball
   a dog chased the ball
   a dog chased the ball
   a dog chased the ball

RL policy        perplexity=1.321  nll/token=0.279
   a dog chased the ball
   a dog chased the ball
   the robot danced gracefully
   a dog chased the ball



## Educational caveats

- Greedy and beam search are decoding algorithms; reinforcement learning is a training or policy-optimization method. This notebook compares their generated outputs only after defining a common evaluator.
- The RL section is tabular and intentionally small. A real RLHF or sequence-level RL setup would learn policy parameters from sampled trajectories, rewards, and baselines.
- Perplexity rewards model likelihood, not necessarily usefulness, truthfulness, safety, or human preference.
- The generated sentences are deliberately simple so the metric and the decoding behavior are the focus.
